# Lab 2 — CIFAR-10 transfer learning
Notebook bàn giao, chưa được chạy Colab. Code nằm trong src/lab_2. Đọc docs/training_handoff.md; chọn GPU T4, điền ba biến dưới rồi chạy lần lượt. Giữ output notebook để làm bằng chứng. Không chạy lại cell train đã hoàn tất hoặc evaluation đã ghi kết quả.

## 1. Mount Drive và thông tin người train

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/Lab 2')
OPERATOR = 'DIEN_HO_TEN'
SESSION_ID = 'DIEN_MA_PHIEN'
assert OPERATOR != 'DIEN_HO_TEN' and SESSION_ID != 'DIEN_MA_PHIEN', 'Điền họ tên và mã phiên duy nhất, ví dụ lab2_20261111_banA_v1'
assert (PROJECT_DIR / 'src/lab_2/training.py').is_file(), 'Sửa PROJECT_DIR cho đúng thư mục Lab 2'
os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = str(PROJECT_DIR / 'src')
os.environ['LAB2_OPERATOR'] = OPERATOR
os.environ['PYTHONUTF8'] = '1'

## 2. Cài thư viện

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 3. GPU và cấu hình phiên — chỉ đổi nơi lưu kết quả

In [ ]:
import copy, re, yaml, torch
assert re.fullmatch(r'[A-Za-z0-9_-]+', SESSION_ID), 'Mã phiên chỉ gồm chữ, số, _ hoặc -'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — chậm; KPI GPU không đo được')
base = yaml.safe_load(Path('configs/lab_2.yaml').read_text(encoding='utf-8'))
config = copy.deepcopy(base)
SESSION_DIR = Path('reports/sessions') / SESSION_ID
config['paths'] = {'checkpoints_dir': str(SESSION_DIR / 'checkpoints'), 'reports_dir': str(SESSION_DIR / 'reports')}
CONFIG_PATH = Path('configs/sessions') / f'{SESSION_ID}.yaml'
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
if CONFIG_PATH.exists():
    assert yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')) == config, 'Config phiên đã khác; giữ phiên cũ và dùng mã phiên mới'
else:
    CONFIG_PATH.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
REPORTS = PROJECT_DIR / config['paths']['reports_dir']
print('Config:', CONFIG_PATH, '\nOutput:', SESSION_DIR)
def run(module, *args):
    subprocess.run([sys.executable, '-m', module, '--config', str(CONFIG_PATH), *args], check=True)

## 4. Kiểm tra offline — dữ liệu giả, không phải số liệu thực nghiệm

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests', 'test.py', '--ignore=tests/test_smoke.py', '-q'], check=True)

## 5. Bước 5 — tải và chia CIFAR-10; torchvision kiểm tra MD5

In [ ]:
import urllib.request
archive = Path('data/cifar-10-python.tar.gz')
archive.parent.mkdir(parents=True, exist_ok=True)
if not archive.exists():
    urllib.request.urlretrieve('https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/resolve/main/cifar-10-python.tar.gz', archive)
run('lab_2.data')

## 6. Bước 2–4 — model pre-trained, kiến trúc và tham số

In [ ]:
run('lab_2.model')

## 7. Smoke thật — 2 batch, 3 cấu hình; chỉ tiếp tục khi đều pass

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_smoke.py', '-q', '-s'], check=True)

## Baseline — resnet18_baseline

In [ ]:
run('lab_2.training', '--run', 'resnet18_baseline')

## Exercise 1 — đổi model — densenet121

In [ ]:
run('lab_2.training', '--run', 'densenet121')

## Exercise 2 — đổi LR — resnet18_lr1e-2

In [ ]:
run('lab_2.training', '--run', 'resnet18_lr1e-2')

## Exercise 2 — đổi batch — resnet18_bs128

In [ ]:
run('lab_2.training', '--run', 'resnet18_bs128')

## Đánh giá test một lần — model được chọn bằng validation

In [ ]:
run('lab_2.evaluation')

## Biểu đồ đánh giá

In [ ]:
from IPython.display import display, Image
for name in ['val_accuracy_by_run.png', 'loss_curve_best.png', 'confusion_matrix.png']:
    display(Image(filename=str(REPORTS / 'figures' / name)))

## Demo — ba ảnh đầu của tập test; có thể dùng --image với ảnh của bạn

In [ ]:
run('lab_2.demo')

## Kết quả demo

In [ ]:
display(Image(filename=str(REPORTS / 'figures/demo_probabilities.png')))

## Bảng kết quả thật và môi trường — nguồn metrics.json của phiên

In [ ]:
import json
report = json.loads((REPORTS / 'metrics.json').read_text(encoding='utf-8'))
for section in ['comparison', 'kpi', 'test_metrics', 'per_class_accuracy']:
    print(section)
    display(report['evaluation'][section])
print('Best run:', report['evaluation']['best_run'])
for name, summary in report['training']['runs'].items():
    print(name, summary['operator'], summary['started_at'], summary['finished_at'], summary['device'], summary['library_versions'], summary['code_commit'])

## Exercise 3 — TensorBoard và bàn giao
Chụp biểu đồ cả 4 run, lưu `tensorboard.png` vào thư mục figures của phiên. Lưu notebook có output. Bàn giao thư mục phiên + config phiên + notebook theo docs/training_handoff.md.

In [ ]:
%load_ext tensorboard
LOGDIR = str(REPORTS / 'runs')
%tensorboard --logdir "$LOGDIR"